# 🔍 ID-VLM — Notebook 02: Baseline Evaluation

**Goal:** Run the un-fine-tuned (zero-shot) Qwen2-VL-2B on identity documents to establish the baseline performance.

This before/after comparison is the core evaluation story:
- **Zero-shot baseline** → this notebook
- **Fine-tuned** → Notebook 04

---

## What this notebook does:
1. Install Unsloth + load Qwen2-VL-2B (no LoRA)
2. Load the test set from Notebook 01
3. Run zero-shot inference on all test images
4. Evaluate with the full harness (CER, exact match, breakdowns)
5. Save baseline predictions for comparison in Notebook 04

## 1. Setup & Install

In [ ]:
%%capture
# Install Unsloth (optimized for Colab T4)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install qwen-vl-utils
!pip install python-Levenshtein

In [ ]:
# Mount Drive & setup paths
from google.colab import drive
drive.mount('/content/drive')

import os, sys, importlib
PROJECT_DIR = '/content/drive/MyDrive/id-vlm'
REPO_DIR = '/content/id-vlm'

if not os.path.exists(f'{REPO_DIR}/config.py'):
    print('Cloning ID-VLM repository...')
    !git clone https://github.com/OmTilwar/ID-VLM.git {REPO_DIR}
else:
    print('Pulling latest repository updates...')
    !git -C {REPO_DIR} pull origin master

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Force reload modules to get latest fixes
if 'src.evaluate' in sys.modules:
    import src.evaluate
    importlib.reload(src.evaluate)
if 'src.dataset' in sys.modules:
    import src.dataset
    importlib.reload(src.dataset)

# Verify test data
test_data_path = f'{PROJECT_DIR}/data/processed/test.jsonl'
assert os.path.exists(test_data_path), \
    f'Test data not found at {test_data_path}! Please run Notebook 01 first.'
print('✅ Setup verified! Test data found.')

In [ ]:
# Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

## 2. Load Model (Zero-Shot, No LoRA)

In [ ]:
from unsloth import FastVisionModel

# Load the base model — NO LoRA adapters
# This is the zero-shot baseline
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Set to inference mode
FastVisionModel.for_inference(model)
print('✅ Qwen2-VL-2B loaded (zero-shot baseline)')
print(f'Model device: {model.device}')

## 3. Load Test Data

In [ ]:
import json
from src.dataset import load_dataset
import config

# Load test set
test_data = load_dataset(f'{PROJECT_DIR}/data/processed/test.jsonl')
print(f'Loaded {len(test_data)} test samples')

# Preview
sample = test_data[0]
print(f'\nSample doc_type: {sample["metadata"]["doc_type"]}')
print(f'Sample answer:   {sample["messages"][1]["content"][0]["text"][:100]}')

## 4. Run Zero-Shot Inference

Run Qwen2-VL on each test image with the extraction prompt.
This is the **un-fine-tuned** model — we expect mediocre performance.

In [ ]:
import time
from PIL import Image
from qwen_vl_utils import process_vision_info

def run_single_inference(model, tokenizer, image_path, prompt):
    """Run a single zero-shot inference."""
    # Load image
    image = Image.open(image_path).convert('RGB')
    
    # Build messages
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': prompt},
            ],
        }
    ]
    
    # Apply chat template
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # Process vision info
    image_inputs, video_inputs = process_vision_info(messages)
    
    # Tokenize
    inputs = tokenizer(
        text=[input_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to(model.device)
    
    # Generate
    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
        )
    elapsed = time.time() - start
    
    # Decode only the generated tokens
    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    output_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return output_text.strip(), elapsed

print('Inference function ready.')

In [ ]:
# Run on all test samples
predictions = []
extraction_prompt = config.EXTRACTION_PROMPT

print(f'Running zero-shot baseline on {len(test_data)} samples...')
print(f'Prompt: "{extraction_prompt[:60]}..."')
print()

total_time = 0
for i, sample in enumerate(test_data):
    metadata = sample.get('metadata', {})
    image_path = metadata.get('image_path', '')
    
    if not os.path.exists(image_path):
        print(f'  ⚠️ Image not found: {image_path}')
        predictions.append({
            'raw_output': '{"error": "image not found"}',
            'metadata': metadata,
            'time_s': 0.0,
            'sample_index': i,
        })
        continue
    
    try:
        output_text, elapsed = run_single_inference(
            model, tokenizer, image_path, extraction_prompt
        )
        total_time += elapsed
        
        predictions.append({
            'raw_output': output_text,
            'metadata': metadata,
            'time_s': elapsed,
            'sample_index': i,
        })
        
        if (i + 1) % 5 == 0 or i == 0:
            print(f'  [{i+1}/{len(test_data)}] {metadata.get("doc_type", "?")} '
                  f'({elapsed:.1f}s) → {output_text[:60]}...')
    
    except Exception as e:
        print(f'  ❌ Error on sample {i}: {e}')
        predictions.append({
            'raw_output': f'{{"error": "{str(e)}"}}',
            'metadata': metadata,
            'time_s': 0.0,
            'sample_index': i,
        })

print(f'\n✅ Done! Total inference time: {total_time:.1f}s')
print(f'Average per sample: {total_time/len(test_data):.1f}s')

## 5. Evaluate Baseline

In [ ]:
import importlib
import src.evaluate
importlib.reload(src.evaluate)

from src.evaluate import (
    evaluate_predictions, format_report_table, save_report,
    parse_vlm_json_output
)

# Build ground truth list
ground_truths = []
for sample in test_data:
    gt_text = sample['messages'][1]['content'][0]['text']
    try:
        fields = json.loads(gt_text)
    except json.JSONDecodeError:
        fields = {}
    ground_truths.append({
        'fields': fields,
        'metadata': sample.get('metadata', {}),
    })

# Run evaluation
baseline_report = evaluate_predictions(predictions, ground_truths)

# Display results
print(format_report_table(baseline_report))

In [ ]:
# Detailed breakdown visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Exact match by document type
ax = axes[0]
doc_types = list(baseline_report['by_doc_type'].keys())
doc_match = [baseline_report['by_doc_type'][dt].get('mean_exact_match', 0) for dt in doc_types]
colors = plt.cm.Set2(np.linspace(0, 1, len(doc_types)))
ax.barh(doc_types, doc_match, color=colors)
ax.set_xlabel('Field Exact Match Rate')
ax.set_title('Baseline: By Document Type')
ax.set_xlim(0, 1)
for i, v in enumerate(doc_match):
    ax.text(v + 0.02, i, f'{v:.1%}', va='center')

# 2. Exact match by capture mode
ax = axes[1]
modes = list(baseline_report['by_capture_mode'].keys())
mode_match = [baseline_report['by_capture_mode'][cm].get('mean_exact_match', 0) for cm in modes]
ax.barh(modes, mode_match, color=['#2196F3', '#4CAF50', '#FF9800'][:len(modes)])
ax.set_xlabel('Field Exact Match Rate')
ax.set_title('Baseline: By Capture Mode')
ax.set_xlim(0, 1)
for i, v in enumerate(mode_match):
    ax.text(v + 0.02, i, f'{v:.1%}', va='center')

# 3. Exact match by field name
ax = axes[2]
fields = list(baseline_report['by_field'].keys())
field_match = [baseline_report['by_field'][fn].get('exact_match_rate', 0) for fn in fields]
# Sort by performance
sorted_pairs = sorted(zip(fields, field_match), key=lambda x: x[1])
fields_sorted, match_sorted = zip(*sorted_pairs) if sorted_pairs else ([], [])
ax.barh(fields_sorted, match_sorted, color='#9C27B0')
ax.set_xlabel('Field Exact Match Rate')
ax.set_title('Baseline: By Field Name')
ax.set_xlim(0, 1)
for i, v in enumerate(match_sorted):
    ax.text(v + 0.02, i, f'{v:.1%}', va='center', fontsize=8)

plt.suptitle('Zero-Shot Qwen2-VL-2B Baseline Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/baseline/baseline_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {PROJECT_DIR}/outputs/baseline/baseline_breakdown.png')

In [ ]:
# Look at specific failure examples
print('=' * 60)
print('FAILURE ANALYSIS: Worst Predictions')
print('=' * 60)

# Find samples with worst CER
sample_results = baseline_report.get('sample_results', [])
worst = sorted(sample_results, key=lambda x: x['field_accuracy']['mean_cer'], reverse=True)[:5]

for i, result in enumerate(worst):
    print(f'\n--- Failure {i+1} ---')
    print(f'Doc type:     {result["doc_type"]}')
    print(f'Capture mode: {result["capture_mode"]}')
    print(f'JSON valid:   {result["json_valid"]}')
    print(f'Mean CER:     {result["field_accuracy"]["mean_cer"]:.3f}')
    print(f'Exact match:  {result["field_accuracy"]["exact_match_rate"]:.1%}')
    print(f'Raw output:   {result["raw_output"][:200]}')
    print()
    for field, data in result['field_accuracy']['per_field'].items():
        status = '✅' if data['match'] else '❌'
        print(f'  {status} {field}: "{data["predicted"]}" vs "{data["ground_truth"]}" (CER={data["cer"]:.3f})')

## 6. Save Baseline Results

In [ ]:
# Save predictions and report
baseline_dir = f'{PROJECT_DIR}/outputs/baseline'
os.makedirs(baseline_dir, exist_ok=True)

# Save raw predictions
with open(f'{baseline_dir}/predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

# Save evaluation report
save_report(baseline_report, f'{baseline_dir}/report.json')

# Save summary
summary = {
    'model': 'Qwen2-VL-2B-Instruct (zero-shot)',
    'n_samples': baseline_report['n_samples'],
    'field_exact_match': baseline_report['overall']['mean_exact_match'],
    'mean_cer': baseline_report['overall']['mean_cer'],
    'document_accuracy': baseline_report['overall']['document_accuracy'],
    'json_parse_rate': baseline_report['json_parse_rate'],
}
with open(f'{baseline_dir}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('✅ Baseline results saved!')
print(f'  Predictions:  {baseline_dir}/predictions.json')
print(f'  Report:       {baseline_dir}/report.json')
print(f'  Summary:      {baseline_dir}/summary.json')
print()
print('Key baseline metrics:')
for k, v in summary.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.3f}')
    else:
        print(f'  {k}: {v}')
print()
print('Proceed to Notebook 03 for fine-tuning!')